# Low-ℓ BB — Source Patch Inspector

Manually enter a source position (RA, Dec) seen in `bb_map_inspect.ipynb` and plot a 3°×3° patch.

**To switch Stokes component**: change `STOKES_KEY` in Paths, re-run from Load map down.  
**To switch mask**: change `ACTIVE_MASK` in Paths, re-run from Load mask down.

In [ ]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

In [ ]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import apply_spt_style, show_map_thumbnail
apply_spt_style()

#### Paths

In [ ]:
# Stokes selector
STOKES_KEY = "T"
_STOKES_FIELD = {"T": 0, "Q": 1, "U": 2}
assert STOKES_KEY in _STOKES_FIELD

# Active mask
ACTIVE_MASK = "mask_250_30"

MASK_FILES = {
    "mask_250_30"   : "puremask8192_0p5medwt_250mJy_30arcmin.npz",
    "mask_250_60"   : "puremask8192_0p5medwt_250mJy_60arcmin.npz",
    "mask_250_nd30" : "puremask8192_0p5medwt_250mJy_nodisk_30arcmin.npz",
    "mask_250_nd60" : "puremask8192_0p5medwt_250mJy_nodisk_60arcmin.npz",
    "mask_100_30"   : "puremask8192_0p5medwt_100mJy_30arcmin.npz",
    "mask_apod_30"  : "puremask8192_0p5medwt_30arcmin.npz",
    "mask_apod_60"  : "puremask8192_0p5medwt_60arcmin.npz",
}

# Data paths
DATA_DIR   = "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix"
COADD_FILE = os.path.join(DATA_DIR, "real_data_maps", "full", "full_220ghz.fits")
MASK_DIR   = "/sptlocal/user/creichardt/bb2020"

# Source patch display
PATCH_DEG   = 3.0
RESO_ARCMIN = 0.5
PATCH_PIX   = int(PATCH_DEG * 60 / RESO_ARCMIN)   # 360 px
CMAP        = "coolwarm"

print(f"Stokes      : {STOKES_KEY}")
print(f"Active mask : {ACTIVE_MASK}")
print(f"Patch       : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px")

#### Load map

In [ ]:
field_idx  = _STOKES_FIELD[STOKES_KEY]
stokes_arr = hp.read_map(COADD_FILE, field=field_idx, partial=False)
nside      = hp.get_nside(stokes_arr)
obs_mask   = np.isfinite(stokes_arr) & (stokes_arr != hp.UNSEEN)

print(f"Loaded : {STOKES_KEY}  from  {os.path.basename(COADD_FILE)}")
print(f"nside  : {nside}   |   observed pixels: {obs_mask.sum():,}")

#### Load mask

In [ ]:
mask_path = os.path.join(MASK_DIR, MASK_FILES[ACTIVE_MASK])
with np.load(mask_path) as d:
    apod = d[d.files[0]].astype(float)

nside_mask = hp.get_nside(apod)
if nside_mask != nside:
    print(f"Downgrading mask nside {nside_mask} → {nside}")
    apod = hp.ud_grade(apod, nside_out=nside)

obs_and_mask = obs_mask & (apod > 0)

stokes_masked                = stokes_arr.copy()
stokes_masked[obs_and_mask] *= apod[obs_and_mask]
stokes_masked[~obs_and_mask] = hp.UNSEEN

print(f"Mask : {ACTIVE_MASK}  ({MASK_FILES[ACTIVE_MASK]})")
print(f"Pixels after mask : {obs_and_mask.sum():,}  (was {obs_mask.sum():,})")

#### Source patch viewer

Enter the RA and Dec of a source (from `bb_map_inspect.ipynb`), run the plot cells, close, repeat.

In [ ]:
# Source coordinates — enter manually
SOURCE_RA  = 0.0   # degrees
SOURCE_DEC = -55.0  # degrees

print(f"Source  →  RA {SOURCE_RA:+.2f}°  Dec {SOURCE_DEC:+.2f}°")

In [ ]:
# Plot unmasked
rms        = float(np.std(stokes_arr[obs_mask]))
vmin, vmax = -3 * rms, 3 * rms

show_map_thumbnail(
    stokes_arr,
    vmin=vmin, vmax=vmax,
    title=f"{STOKES_KEY}  unmasked  |  RA {SOURCE_RA:+.2f}°  Dec {SOURCE_DEC:+.2f}°",
    unit="Tcmb", cmap=CMAP,
    rot=(SOURCE_RA, SOURCE_DEC, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Plot masked
rms_m      = float(np.std(stokes_masked[obs_and_mask]))
vmin, vmax = -3 * rms_m, 3 * rms_m

show_map_thumbnail(
    stokes_masked,
    vmin=vmin, vmax=vmax,
    title=f"{STOKES_KEY}  masked ({ACTIVE_MASK})  |  RA {SOURCE_RA:+.2f}°  Dec {SOURCE_DEC:+.2f}°",
    unit="Tcmb", cmap=CMAP,
    rot=(SOURCE_RA, SOURCE_DEC, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")